# IBL Brain Wide Map → HIPPIE CSV converter

Download a single IBL Brain Wide Map (BWM) insertion via the [ONE API](https://int-brain-lab.github.io/iblenv/) and convert it into the canonical HIPPIE CSV layout (`waveforms.csv`, `isi_dist.csv`, `acg.csv`, `labels.csv`, `metadata.csv`).

Mirrors `allen_nwb_to_csv_converter.ipynb`; uses the same `Neurocurator` class for the ISI / ACG / waveform-feature computations so the outputs are byte-for-byte compatible with the rest of the HIPPIE pipeline.

Data source: openalyx.internationalbrainlab.org (public Brain Wide Map mirror).

## What this notebook does

1. **Setup**: connects anonymously to the public Open Alyx instance.
2. **Session selection**: takes a single `EID` (insertion ID) or lists Brain Wide Map insertions if you leave it blank.
3. **Download**: pulls `spikes`, `clusters`, and `channels` ALF objects for one probe.
4. **Quality filter** (optional): keeps only IBL `good` units (`clusters.label == 1`).
5. **Process**: builds a `Neurocurator` instance manually and computes ISI distributions, autocorrelograms, and waveform-shape features.
6. **Export**: writes the canonical 5-CSV bundle to `./ibl_<EID>_neurocurator_csv/`.

Labels (`labels.csv`) default to Allen CCF brain-region acronyms — matching `allen_scope_neuropixel_area_subset`. Swap in your own label column at the bottom if you want cell-type labels instead.

## Setup

Install the IBL deps once via the project's `[ibl]` extra (from the repo root):

```bash
pip install -e ".[ibl]"
```

Or, if you don't have the HIPPIE repo cloned:

```bash
pip install ONE-api iblatlas
```

`iblatlas` is optional but recommended: it lets the notebook map IBL's CCF region IDs to Allen acronyms (e.g., `184 → "FRP"`) for `labels.csv`. Without it, `labels.csv` falls back to the raw integer region IDs.

No credentials needed — Open Alyx is read-only public.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Neurocurator from the same directory as this notebook.
sys.path.append(str(Path.cwd()))
from neurocurator import Neurocurator  # noqa: E402

## Configuration

In [ ]:
# IBL insertion ID (eid) -- one Neuropixel probe insertion in one session.
# Set to None to browse Brain Wide Map insertions instead.
EID = "8c732bf2-639d-496c-bf82-464bc9c2d54b"  # MFD_09 2023-10-19, ~16 MB clusters.waveforms

# Restrict to IBL 'good' units (clusters.metrics.label == 1: ISI violations
# < 0.5, presence ratio >= 0.5, minimum 100 spikes). The paper Methods use
# this filter for Brain Wide Map.
GOOD_ONLY = True

# IBL Neuropixels 1.0 / 2.0 sampling rate (Hz)
SAMPLING_RATE = 30_000

# Output directory will be created if it does not exist.
OUTPUT_DIR = f"./ibl_{EID[:8] if EID else 'session'}_neurocurator_csv"

# ONE cache (downloads parquet metadata + per-object ALF tables here).
ONE_CACHE_DIR = "./ibl_cache"
os.makedirs(ONE_CACHE_DIR, exist_ok=True)

## Connect to Open Alyx

In [ ]:
from one.api import ONE

# Public read-only mirror; password is documented as 'international'.
# Passing cache_dir on the ONE(...) call (instead of via ONE.setup) keeps the
# cache local to this notebook -- ONE.setup would write a persistent config
# in ~/.one_params/ that other ONE-using code on this machine would inherit.
one = ONE(
    base_url="https://openalyx.internationalbrainlab.org",
    password="international",
    silent=True,
    cache_dir=ONE_CACHE_DIR,
)
print(f"Connected to {one.alyx.base_url}")

## Pick an insertion

If `EID` is not set, list a few Brain Wide Map insertions so you can copy one.

In [ ]:
if EID is None:
    # Brain Wide Map released insertions live in the 'Brainwidemap' tag.
    insertions = one.alyx.rest(
        "insertions", "list", django="session__projects__name__icontains,brainwide"
    )
    print(f"Found {len(insertions)} Brain Wide Map insertions.")
    df = pd.DataFrame(
        [
            {
                "eid": ins["id"],
                "subject": ins["session_info"]["subject"],
                "date": ins["session_info"]["start_time"][:10],
                "probe": ins["name"],
                "lab": ins["session_info"]["lab"],
            }
            for ins in insertions[:20]
        ]
    )
    print(df.to_string(index=False))
    print("\nSet EID in the configuration cell above and re-run.")
else:
    info = one.alyx.rest("insertions", "read", id=EID)
    print(f"EID:     {EID}")
    print(f"Subject: {info['session_info']['subject']}")
    print(f"Date:    {info['session_info']['start_time'][:10]}")
    print(f"Probe:   {info['name']}")
    print(f"Lab:     {info['session_info']['lab']}")

## Download `spikes`, `clusters`, `channels`

Each ALF object is a dict of numpy arrays keyed by attribute. We pull from the `pykilosort` collection, which is the standard BWM upload.

In [ ]:
assert EID is not None, "Set EID before running this cell."

collection = f"alf/{info['name']}/pykilosort"
session_eid = info["session"]

spikes = one.load_object(session_eid, "spikes", collection=collection)
clusters = one.load_object(session_eid, "clusters", collection=collection)
channels = one.load_object(session_eid, "channels", collection=collection)

n_clusters = len(clusters["channels"])
print(f"Loaded {n_clusters} clusters, {len(spikes['times']):,} total spikes")
print(f"spikes  keys: {sorted(spikes.keys())}")
print(f"clusters keys: {sorted(clusters.keys())}")
print(f"channels keys: {sorted(channels.keys())}")

## Quality filter

IBL ships a single-scalar quality label per cluster, stored inside the `clusters.metrics` DataFrame: `label == 1.0` means "good" (passes ISI-violation, presence-ratio, and minimum-spike-count thresholds). Set `GOOD_ONLY = False` to keep all units.

In [ ]:
metrics = clusters.get("metrics")
if GOOD_ONLY and metrics is not None and "label" in metrics.columns:
    keep_mask = np.asarray(metrics["label"]) >= 1.0
    keep_ids = np.where(keep_mask)[0]
    print(f"Keeping {keep_mask.sum()} / {len(keep_mask)} clusters labeled 'good'.")
else:
    keep_ids = np.arange(n_clusters)
    print(f"Keeping all {n_clusters} clusters (no quality filter applied).")

## Build per-unit spike trains and waveforms

In [ ]:
# Bucket spike times by cluster, in milliseconds (Neurocurator's convention).
spike_times_s = np.asarray(spikes["times"])
spike_clusters = np.asarray(spikes["clusters"])

spike_times_train = []
for cid in keep_ids:
    times_ms = spike_times_s[spike_clusters == cid] * 1000.0
    spike_times_train.append(np.sort(times_ms))

print(f"Built spike trains for {len(spike_times_train)} units.")
print(f"Median spike count per unit: {int(np.median([len(s) for s in spike_times_train]))}")

In [ ]:
# Extract per-unit mean waveforms from clusters.waveforms.
# IBL stores (n_clusters, n_samples, n_channels). We pick the channel with
# the largest peak-to-peak amplitude, then trough-center to 50 samples
# (20 pre, 30 post) -- matching neurocurator.extract_waveforms.
N = 50
BEFORE = int(N * 2 / 5)  # 20
AFTER = N - BEFORE       # 30

raw_waveforms = np.asarray(clusters["waveforms"])  # (n_clusters, T, n_ch)

wf_rows = []
for cid in keep_ids:
    wf = raw_waveforms[cid]
    if wf.ndim == 2:
        # pick the channel with the largest amplitude (max - min)
        ch_idx = int(np.argmax(wf.max(axis=0) - wf.min(axis=0)))
        wf1d = wf[:, ch_idx]
    else:
        wf1d = wf
    if wf1d.size == 0 or not np.isfinite(wf1d).any():
        wf_rows.append(np.zeros(N, dtype=np.float32))
        continue
    min_idx = int(np.nanargmin(wf1d))
    lo = max(0, min_idx - BEFORE)
    seg = wf1d[lo : lo + N]
    if seg.size < N:
        seg = np.pad(seg, (0, N - seg.size))
    wf_rows.append(seg.astype(np.float32))

waveforms_df = pd.DataFrame(wf_rows)
print(f"Waveforms shape: {waveforms_df.shape}")

## Run Neurocurator (ISI distribution, ACG, shape features)

In [ ]:
neurocurator = Neurocurator()
neurocurator.spike_times_train = spike_times_train
neurocurator.sampling_rate = SAMPLING_RATE
neurocurator.waveforms = waveforms_df

# Spike times for the ACG call need to be in milliseconds and sorted -- already true.
neurocurator.isi_distribution = neurocurator.compute_isi_distribution(time_window=100)
neurocurator.acgs = neurocurator.compute_autocorrelogram(neurocurator.spike_times_train)

print(f"ISI: {neurocurator.isi_distribution.shape}, ACG: {neurocurator.acgs.shape}")

In [ ]:
# Metadata: channel position, brain region, IBL quality metrics, plus the
# Neurocurator-computed waveform-shape features.

# Channel index per kept cluster, and 2D probe coordinates.
ch_idx_per_unit = np.asarray(clusters["channels"])[keep_ids]
ch_xy = np.asarray(channels["localCoordinates"]) if "localCoordinates" in channels else None
if ch_xy is not None:
    x_coord = ch_xy[ch_idx_per_unit, 0]
    y_coord = ch_xy[ch_idx_per_unit, 1]
else:
    x_coord = np.zeros(len(keep_ids))
    y_coord = np.zeros(len(keep_ids))

# Brain region: IBL stores CCF region IDs per channel; map to acronyms via
# iblatlas. Fall back to the raw integer IDs if iblatlas is not installed.
ccf_ids_per_channel = channels.get("brainLocationIds_ccf_2017")
if ccf_ids_per_channel is not None:
    ccf_ids = np.asarray(ccf_ids_per_channel)[ch_idx_per_unit]
    try:
        from iblatlas.regions import BrainRegions
        br = BrainRegions()
        id_to_acronym = dict(zip(br.id.astype(int).tolist(), br.acronym.tolist()))
        region = np.array([id_to_acronym.get(int(i), str(int(i))) for i in ccf_ids], dtype=object)
    except ImportError:
        print("iblatlas not installed -- writing raw CCF region IDs as labels.")
        region = ccf_ids.astype(str)
else:
    region = np.full(len(keep_ids), "unknown", dtype=object)

metadata = pd.DataFrame({
    "unit_id": keep_ids,
    "x": x_coord,
    "y": y_coord,
    "acronym": region,
})

# Carry forward IBL quality metrics from the clusters.metrics DataFrame.
if metrics is not None:
    metrics_kept = metrics.iloc[keep_ids].reset_index(drop=True)
    for col in ("firing_rate", "presence_ratio", "amp_median", "spike_count", "label"):
        if col in metrics_kept.columns:
            metadata[col] = metrics_kept[col].values

neurocurator.metadata_obs = metadata
neurocurator.compute_all_waveform_features()
neurocurator.compute_firing_rate()
try:
    neurocurator.compute_minimum_isi()
except ValueError:
    pass  # units with < 2 spikes blow up np.diff().min()

neurocurator.validate_data_integrity()
print(f"Metadata columns: {list(neurocurator.metadata_obs.columns)}")

## Write CSVs

Writes the canonical HIPPIE layout into `OUTPUT_DIR`:

```
ibl_<EID>_neurocurator_csv/
├── waveforms.csv    # n_units × 50 timepoints
├── isi_dist.csv     # n_units × 100 ms bins
├── acg.csv          # n_units × 201 ms bins (-100..+100 ms)
├── labels.csv       # n_units × 1  (brain-region acronym; rename if you have cell types)
├── metadata.csv     # per-unit features
└── session_summary.txt
```

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

neurocurator.waveforms.to_csv(os.path.join(OUTPUT_DIR, "waveforms.csv"), index=False)
neurocurator.isi_distribution.to_csv(os.path.join(OUTPUT_DIR, "isi_dist.csv"), index=False)
neurocurator.acgs.to_csv(os.path.join(OUTPUT_DIR, "acg.csv"), index=False)
neurocurator.metadata_obs.to_csv(os.path.join(OUTPUT_DIR, "metadata.csv"), index=False)

# labels.csv: brain-region acronyms by default. Replace with cell-type
# labels here if you have them (e.g., from a separate spreadsheet keyed
# on unit_id).
pd.DataFrame({"label": neurocurator.metadata_obs["acronym"].astype(str)}).to_csv(
    os.path.join(OUTPUT_DIR, "labels.csv"), index=False
)

with open(os.path.join(OUTPUT_DIR, "session_summary.txt"), "w") as f:
    f.write(f"IBL Brain Wide Map insertion {EID}\n")
    f.write(f"Subject: {info['session_info']['subject']}\n")
    f.write(f"Date:    {info['session_info']['start_time'][:10]}\n")
    f.write(f"Probe:   {info['name']}\n")
    f.write(f"Lab:     {info['session_info']['lab']}\n")
    f.write(f"Sampling rate: {SAMPLING_RATE} Hz\n")
    f.write(f"Quality filter: {'IBL good (label==1)' if GOOD_ONLY else 'none'}\n")
    f.write(f"Units exported: {len(neurocurator.spike_times_train)}\n")
    f.write(f"Waveform points: {neurocurator.waveforms.shape[1]}\n")
    f.write(f"ISI bins: {neurocurator.isi_distribution.shape[1]}\n")
    f.write(f"ACG bins: {neurocurator.acgs.shape[1]}\n")

print(f"Wrote 5 CSVs + summary to {OUTPUT_DIR}")
print("\nNext step: copy or symlink the folder into datasets_hippie/:")
print(f"  mv {OUTPUT_DIR} ../datasets_hippie/ibl_{EID[:8]}")
print("Then validate:")
print(f"  hippie-cli validate-data ../datasets_hippie/ibl_{EID[:8]}")

## (Optional) Sanity-check plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))

for i in range(min(10, len(neurocurator.waveforms))):
    axes[0].plot(neurocurator.waveforms.iloc[i], alpha=0.6)
axes[0].set(title="First 10 waveforms", xlabel="sample", ylabel="amplitude")

axes[1].imshow(neurocurator.isi_distribution.values[:50], aspect="auto", origin="lower")
axes[1].set(title="ISI distribution (50 units)", xlabel="bin (ms)", ylabel="unit")

axes[2].imshow(neurocurator.acgs.values[:50], aspect="auto", origin="lower")
axes[2].set(title="ACG (50 units)", xlabel="lag bin", ylabel="unit")

plt.tight_layout()
plt.show()